In [1]:
# YOLO - PASCAL VOC 2007 다운로드 / 압축 해제 (Windows 호환, 중단 후 재실행 안전)
import os, sys, urllib.request, tarfile, subprocess

urls = {
    "VOCtrainval_06-Nov-2007.tar": "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
    "VOCtest_06-Nov-2007.tar":     "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar",
}

def download(name, url):
    # 서버가 알려주는 전체 크기
    with urllib.request.urlopen(urllib.request.Request(url, method="HEAD")) as r:
        remote = int(r.headers.get("Content-Length", 0))
    if os.path.exists(name) and remote and os.path.getsize(name) == remote:
        print(f"[skip] {name} ({remote/1e6:.1f} MB, 이미 완료)")
        return
    tmp = name + ".part"
    print(f"[download] {url}")
    def _hook(blk, blk_size, total):
        if total > 0:
            print(f"\r  {min(blk*blk_size/total*100, 100):5.1f}%", end="")
    urllib.request.urlretrieve(url, tmp, _hook)
    if remote and os.path.getsize(tmp) != remote:
        os.remove(tmp)
        raise IOError(f"{name}: 다운로드 불완전 ({os.path.getsize(tmp)}/{remote}) - 셀을 다시 실행하세요")
    os.replace(tmp, name)
    print(f"\r[done] {name} ({os.path.getsize(name)/1e6:.1f} MB)")

for name, url in urls.items():
    download(name, url)

for name, outdir in [("VOCtrainval_06-Nov-2007.tar", "VOCtrainval_2007"),
                     ("VOCtest_06-Nov-2007.tar", "VOCtest_2007")]:
    os.makedirs(outdir, exist_ok=True)
    print(f"[extract] {name} -> {outdir}/")
    with tarfile.open(name) as t:
        t.extractall(outdir)
print("[extract] complete")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xmltodict"], check=True)
print("xmltodict ready")


[skip] VOCtrainval_06-Nov-2007.tar (460.0 MB, 이미 완료)
[skip] VOCtest_06-Nov-2007.tar (451.0 MB, 이미 완료)
[extract] VOCtrainval_06-Nov-2007.tar -> VOCtrainval_2007/
[extract] VOCtest_06-Nov-2007.tar -> VOCtest_2007/
[extract] complete
xmltodict ready


In [2]:
import numpy as np
import cv2
import xmltodict
import tensorflow as tf

from tqdm import tqdm
from glob import glob
from tensorflow.keras.callbacks import ModelCheckpoint

import warnings
warnings.filterwarnings("ignore")

In [3]:
train_x_path = './VOCtrainval_2007/VOCdevkit/VOC2007/JPEGImages'
train_y_path = './VOCtrainval_2007/VOCdevkit/VOC2007/Annotations'
test_x_path = './VOCtest_2007/VOCdevkit/VOC2007/JPEGImages'
test_y_path = './VOCtest_2007/VOCdevkit/VOC2007/Annotations'

In [4]:
image_file_path_list = sorted([x for x in glob(train_x_path + "/**")])
subset_size = len(image_file_path_list) // 70
image_file_path_list = image_file_path_list[:subset_size]

xml_file_path_list = sorted([x for x in glob(train_y_path + "/**")])
xml_file_path_list = xml_file_path_list[:subset_size]

test_image_file_path_list = sorted([x for x in glob(test_x_path + "/**")])
subset_size = len(test_image_file_path_list) // 70
test_image_file_path_list = test_image_file_path_list[:subset_size]

test_xml_file_path_list = sorted([x for x in glob(test_y_path + "/**")])
test_xml_file_path_list = test_xml_file_path_list[:subset_size]

In [5]:
def get_classes_in_image(xml_file_list):
    classes_in_data_set = set()

    for xml_file_path in xml_file_list:
        with open(xml_file_path, "r") as file:
            xml_file = xmltodict.parse(file.read())
            objects = xml_file["annotation"]["object"]

            if not isinstance(objects, list):
                objects = [objects]

            for obj in objects:
                classes_in_data_set.add(obj["name"].lower())

    classes_in_data_set = sorted(classes_in_data_set)
    print(classes_in_data_set)

    return classes_in_data_set

classes_inDataSet = get_classes_in_image(test_xml_file_path_list)

['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']


In [6]:
def get_label_from_image(xml_file_path, Classes_inDataSet):

    def transform_coordinates(coordinates, Image_Width, Image_Height):
        x_min, y_min, x_max, y_max = coordinates
        x_min, x_max = [(224.0 / Image_Width) * x for x in [x_min, x_max]]
        y_min, y_max = [(224.0 / Image_Height) * x for x in [y_min, y_max]]

        x, y, w, h = (x_min + x_max) / 2.0, (y_min + y_max) / 2.0, (x_max - x_min) / 224.0, (y_max - y_min) / 224.0

        return x, y, w, h

    with open(xml_file_path, "r") as f:
        xml_file = xmltodict.parse(f.read())

    Image_Height, Image_Width = (float(xml_file["annotation"]["size"]["height"]), float(xml_file["annotation"]["size"]["width"]))
    label = np.zeros((7, 7, 25), dtype = float)
    objects = xml_file["annotation"]["object"]

    if not isinstance(objects, list): objects = [objects]

    for obj in objects:
        class_index = Classes_inDataSet.index(obj["name"].lower())
        coordinates = (float(obj["bndbox"]["xmin"]), float(obj["bndbox"]["ymin"]), float(obj["bndbox"]["xmax"]), float(obj["bndbox"]["ymax"]))
        x, y, w, h = transform_coordinates(coordinates, Image_Width, Image_Height)

        x_cell, y_cell = int(x / 32), int(y / 32)
        x_val_inCell, y_val_inCell = (x - x_cell * 32.0) / 32.0, (y - y_cell * 32.0) / 32.0
        class_index_inCell = class_index + 5
        label[y_cell, x_cell,  :5] = [x_val_inCell, y_val_inCell, w, h, 1.0]
        label[y_cell, x_cell, class_index_inCell] = 1.0

    return label

In [7]:
def make_dataset(image_file_path_list, xml_file_path_list, Classes_inDataSet):

    def process_image(image_file_path):
        image = cv2.imread(image_file_path)

        return cv2.resize(image, (224, 224)) / 255.0

    image_dataset = [process_image(image_path) for image_path in tqdm(image_file_path_list, desc = "Processing Images")]
    label_dataset = [get_label_from_image(xml_path, Classes_inDataSet) for xml_path in tqdm(xml_file_path_list)]

    image_dataset = np.array(image_dataset, dtype = np.float32)
    label_dataset = np.array(label_dataset, dtype = np.float32).reshape(-1, 7, 7, 25)

    return tf.convert_to_tensor(image_dataset), tf.convert_to_tensor(label_dataset)

train_image_dataset, train_label_dataset = make_dataset(image_file_path_list, xml_file_path_list, classes_inDataSet)
test_image_dataset, test_label_dataset = make_dataset(test_image_file_path_list, test_xml_file_path_list, classes_inDataSet)

100%|██████████| 70/70 [00:00<00:00, 5584.43it/s]


In [8]:
max_num = len(tf.keras.applications.VGG16(weights = "imagenet", include_top = False, input_shape = (224, 224, 3)).layers)

YOLO = tf.keras.models.Sequential(name = "YOLO")

for i in range(0, max_num - 1):
    YOLO.add(tf.keras.applications.VGG16(weights = "imagenet", include_top = False, input_shape = (224, 224, 3)).layers[i])

initializer = tf.keras.initializers.RandomNormal(mean = 0.0, stddev = 0.01)
leaky_relu = tf.keras.layers.LeakyReLU(alpha = 0.01)
regularizer = tf.keras.regularizers.l2(0.0005)

for layer in YOLO.layers:
    layer.trainable = False

    if (hasattr(layer, "activation")) == True:
        layer.activation = leaky_relu

In [9]:
def add_conv_layer(YOLO, filters, name):
    YOLO.add(tf.keras.layers.Conv2D(filters, (3, 3), activation = leaky_relu, kernel_initializer = initializer,
                                    kernel_regularizer = regularizer, padding = "SAME", name = name, dtype = "float32"))

def add_dense_layer(YOLO, units ,name, activation = leaky_relu, dropout = None):
    YOLO.add(tf.keras.layers.Dense(units, activation = leaky_relu, kernel_initializer = initializer,
                                   kernel_regularizer = regularizer, name = name, dtype = "float32"))

    if dropout:
        YOLO.add(tf.keras.layers.Dropout(dropout))

add_conv_layer(YOLO, 1024, "detection_conv1")
add_conv_layer(YOLO, 1024, "detection_conv2")
YOLO.add(tf.keras.layers.MaxPool2D((2, 2)))
add_conv_layer(YOLO, 1024, "detection_conv3")
add_conv_layer(YOLO, 1024, "detection_conv4")

YOLO.add(tf.keras.layers.Flatten())
add_dense_layer(YOLO, 4096, "detection_linear1", dropout = 0.5)
add_dense_layer(YOLO, 1470, "detection_linear2", activation = None)

YOLO.add(tf.keras.layers.Reshape((7, 7, 30), name = "output", dtype = "float32"))


In [10]:
def yolo_multitask_loss(y_true, y_pred):
    batch_loss = 0

    for true_vals, pred_vals in zip(y_true, y_pred):
        true_vals = tf.reshape(true_vals, [49, 25])
        pred_vals = tf.reshape(pred_vals, [49, 30])
        cell_losses = []

        for true_cell, pred_cell in zip(true_vals, pred_vals):
            bbox1_pred, bbox1_confidence = pred_cell[:4], pred_cell[4]
            bbox2_pred, bbox2_confidence, class_pred = pred_cell[5:9], pred_cell[9], pred_cell[10:]

            bbox_true, bbox_true_confidence = true_cell[:4], true_cell[4]
            class_true = true_cell[5:]

            def calculate_iou(bbox_pred, bbox_true):
                pred_area = bbox_pred[2] * bbox_pred[3]
                true_area = bbox_true[2] * bbox_true[3]
                pred_minmax = [bbox_pred[0] - 0.5 * bbox_pred[2],
                               bbox_pred[1] - 0.5 * bbox_pred[3],
                               bbox_pred[0] + 0.5 * bbox_pred[2],
                               bbox_pred[1] + 0.5 * bbox_pred[3]]
                
                true_minmax = [bbox_true[0] - 0.5 * bbox_true[2],
                               bbox_true[1] - 0.5 * bbox_true[3],
                               bbox_true[0] + 0.5 * bbox_true[2],
                               bbox_true[1] + 0.5 * bbox_true[3]]

                inter_xy_min = tf.maximum(pred_minmax[:2], true_minmax[:2])
                inter_xy_max = tf.minimum(pred_minmax[2:], true_minmax[2:])

                inter_area = tf.maximum(0.0, inter_xy_max[0] - inter_xy_min[0]) * tf.maximum(0.0, inter_xy_max[1] - inter_xy_min[1])        
                union_area = pred_area + true_area - inter_area 

                return inter_area / union_area

            iou_bbox1 = calculate_iou(bbox1_pred, bbox_true)            
            iou_bbox2 = calculate_iou(bbox2_pred, bbox_true)       

            responsible_bbox = bbox1_pred if iou_bbox1 > iou_bbox2 else bbox2_pred
            responsible_bbox_confidence = bbox1_confidence if iou_bbox1 > iou_bbox2 else bbox2_confidence
            non_responsible_bbox_confidence = bbox2_confidence if iou_bbox1 > iou_bbox2 else bbox1_confidence

            obj_exist = 1.0 - tf.cast(tf.reduce_all(tf.equal(bbox_true, 0.0)), tf.float32)

            localization_err = tf.reduce_sum(tf.square(bbox_true - responsible_bbox)) * obj_exist
            confidence_err_obj = tf.square(responsible_bbox_confidence - bbox_true_confidence) * obj_exist
            confidence_err_noobj = 0.5 * tf.square(non_responsible_bbox_confidence) * (1.0 - obj_exist)
            classification_err = tf.reduce_sum(tf.square(class_true - class_pred)) * obj_exist

            cell_loss = 5.0 * localization_err + confidence_err_obj + confidence_err_noobj + classification_err
            cell_losses.append(cell_loss)

        batch_loss += tf.reduce_sum(cell_losses)

    batch_loss /= tf.cast(tf.shape(y_true)[0], tf.float32)

    return batch_loss

In [11]:
BATCH_SIZE = 64
EPOCHS = 120
SAVE_PATH = 'yolo.h5'

def lr_schedule(epoch):
    if epoch < 75:
        return 0.001 + 0.001 * (epoch / 75.0)

    elif epoch < 105:
        return 0.001

    else:
        return 0.0001

def compile_and_train_model(model, train_data, val_data):
    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        SAVE_PATH, verbose = 1, save_best_only = True
    )

    lr_callback = tf.keras.callbacks.LearningRateScheduler(lr_schedule)
    optimizer = tf.keras.optimizers.SGD(learning_rate = 0.001, momentum = 0.9)

    model.compile(loss = yolo_multitask_loss, optimizer = optimizer, run_eagerly = True)

    model.fit(train_data[0], train_data[1], batch_size = BATCH_SIZE, validation_data = val_data, epochs = EPOCHS, verbose = 1,
              callbacks = [checkpoint_callback, lr_callback])

In [ ]:
compile_and_train_model(YOLO, (train_image_dataset, train_label_dataset), (test_image_dataset, test_label_dataset))

Epoch 1/120

2/2 [==============================] - ETA: 0s - loss: 30.0595  
Epoch 1: val_loss improved from inf to 26.71985, saving model to yolo.h5
2/2 [==============================] - 121s 52s/step - loss: 30.0595 - val_loss: 26.7199 - lr: 0.0010
Epoch 2/120
1/2 [==============>...............] - ETA: 43s - loss: 25.5090

In [ ]:
IMAGE_SIZE = (224, 224)
CELL_SIZE = 32

def convert_to_corner_coordinates(x, y, bbox, image_size):
    bbox_x = (CELL_SIZE * x + bbox[0] * CELL_SIZE) * image_size[0] / IMAGE_SIZE[0]
    bbox_y = (CELL_SIZE * y + bbox[1] * CELL_SIZE) * image_size[1] / IMAGE_SIZE[1]
    bbox_w = bbox[2] * image_size[0]
    bbox_h = bbox[3] * image_size[1]

    min_x = int(bbox_x - bbox_w/2)
    min_y = int(bbox_y - bbox_h/2)
    max_x = int(bbox_x + bbox_w/2)
    max_y = int(bbox_y + bbox_h/2)

    return [min_x, min_y, max_x, max_y]

def process_single_bbox(x, y, bbox, image_size, classes_score, class_names):
    idx_highest_score = np.argmax(classes_score)
    highest_score = classes_score[idx_highest_score]
    highest_score_name = class_names[idx_highest_score]

    corner_coords = convert_to_corner_coordinates(x, y, bbox, image_size)

    return corner_coords + [highest_score, highest_score_name]

def nms(bbox_list, threshold=0.6):
    return [bbox for bbox in bbox_list if bbox[4] > threshold]

def get_YOLO_output(YOLO, image_path, class_names):
    image_cv = cv2.imread(image_path)
    original_h, original_w, _ = image_cv.shape
    image_resized = cv2.resize(image_cv, IMAGE_SIZE) / 255.0
    image_input = np.expand_dims(image_resized, axis=0).astype('float32')

    yolo_output = YOLO(image_input)[0].numpy()

    bbox_list = []
    for y in range(7):
        for x in range(7):
            bbox1 = yolo_output[y][x][:4]
            bbox2 = yolo_output[y][x][5:9]
            bbox1_score = yolo_output[y][x][10:] * yolo_output[y][x][4]
            bbox2_score = yolo_output[y][x][10:] * yolo_output[y][x][9]

            bbox1_processed = process_single_bbox(x, y, bbox1, (original_w, original_h), bbox1_score, class_names)
            bbox2_processed = process_single_bbox(x, y, bbox2, (original_w, original_h), bbox2_score, class_names)

            bbox_list.extend([bbox1_processed, bbox2_processed])

    nms_boxes = nms(bbox_list)

    for bbox in nms_boxes:
        cv2.rectangle(image_cv, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (255, 0, 0), 2)

    cv2.imwrite('output.jpg', image_cv)

In [ ]:
get_YOLO_output(YOLO,'./data/like_lenna.png',classes_inDataSet)